In [ ]:
import dataiku
import pandas as pd

In [ ]:
# Replace with the name of the S3 connection you are migrating away from.
old_s3_connection = "personal_AWS_S3"
# Replace with the name of the target S3 connection to migrate datasets to.
new_s3_connection = "dataiku-managed-storage"


In [ ]:
# Initialize the Dataiku API client
client = dataiku.api_client()

# Get the current project
project = client.get_default_project()

# List all datasets in the current project
unique_dataset_names = set(dataset['name'] for dataset in project.list_datasets())

In [ ]:
def get_dataset_configs() -> "pd.DataFrame":
    """Return a DataFrame of dataset type, connection, and path info for the current project."""
    extracted_data = [
        {
            'type': row.get('type'),
            'connection': row.get('params', {}).get('connection'),
            'name': row.get('name'),
            'table': row.get('params', {}).get('table'),
            'catalog': row.get('params', {}).get('catalog'),
            'schema': row.get('params', {}).get('schema'),
            'path': row.get('params', {}).get('path'),
        }
        for row in project.list_datasets()
    ]
    return pd.DataFrame(extracted_data).sort_values(by=['type', 'connection', 'name'])

In [ ]:
get_dataset_configs()

In [ ]:
for dataset_name in unique_dataset_names:
    dataset_handle = project.get_dataset(dataset_name)
    settings = dataset_handle.get_settings()
    raw = settings.get_raw()
    if raw.get('type') == 'S3' and raw.get('params', {}).get('connection') == old_s3_connection:
        # Update to the new S3 connection
        settings.set_connection_and_path(new_s3_connection, settings.get_raw_params()['path'])
        settings.save()
        print(f"Dataset {dataset_name} updated to use connection: {new_s3_connection}")


In [ ]:
get_dataset_configs()